# 🚀 CIFAR-100 Training on Colab

自動掛載 Google Drive、clone repo、偵測 checkpoint 並恢復訓練。

| Step | 說明 | 執行時機 |
|------|------|----------|
| 1 | 掛載 Google Drive | 每次啟動 |
| 2 | Clone / Pull 程式碼 | 每次啟動 |
| 3 | 安裝依賴套件 | 每次啟動 |
| 4 | 確認 GPU 環境 | 每次啟動 |
| 5 | 自動偵測 Checkpoint 並訓練 | 訓練時 |
| 6 | 確認 Checkpoint 已存到 Drive | 訓練後 |
| 7 | 執行模型評量 | 訓練完成後 |
| 8 | 清除舊 Checkpoint（選用） | 需重頭訓練時 |

## Step 1｜掛載 Google Drive

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

import os

CHECKPOINT_DIR = '/content/drive/MyDrive/cifar100_checkpoints'
os.makedirs(CHECKPOINT_DIR, exist_ok=True)
print(f'✅ Checkpoint 目錄：{CHECKPOINT_DIR}')

## Step 2｜Clone / Pull 最新程式碼

In [ ]:
import os

!git config --global user.name "rvgoing"
!git config --global user.email "cmkdkimo@gmail.com"

GITHUB_REPO = 'https://github.com/rvgoing/CIFARMLOps.git'
REPO_NAME   = 'CIFARMLOps'

if os.path.exists(f'/content/{REPO_NAME}'):
    print('📦 Repo 已存在，執行 git pull ...')
    %cd /content/{REPO_NAME}
    !git pull
else:
    print('📦 Clone repo ...')
    !git clone {GITHUB_REPO}
    %cd /content/{REPO_NAME}

print('✅ 程式碼已是最新版本')

## Step 3｜安裝依賴套件

In [ ]:
import subprocess
import sys

print('📦 開始安裝套件...')

with open('requirements.txt') as f:
    packages = [line.strip() for line in f if line.strip() and not line.startswith('#')]

for pkg in packages:
    print(f'  安裝 {pkg} ...', end=' ')
    result = subprocess.run(
        [sys.executable, '-m', 'pip', 'install', '-q', pkg],
        capture_output=True, text=True
    )
    if result.returncode == 0:
        print('✅')
    else:
        print('❌')
        print(result.stderr)

print('\n✅ 所有套件安裝完成')

## Step 4｜確認 GPU 環境

In [ ]:
import torch

print(f'PyTorch 版本：{torch.__version__}')
print(f'CUDA 可用：{torch.cuda.is_available()}')

if torch.cuda.is_available():
    print(f'GPU：{torch.cuda.get_device_name(0)}')
    print(f'VRAM：{torch.cuda.get_device_properties(0).total_memory / 1024**3:.1f} GB')
else:
    print('⚠️  未偵測到 GPU，請確認 Runtime > Change runtime type > T4 GPU')

!nvidia-smi

## Step 5｜自動偵測 Checkpoint 並開始訓練

> 自動判斷是否有 `checkpoint.pth` 或 `model_best.pth`，有則恢復訓練，無則從頭開始。

In [ ]:
import os

# 確保變數在此 cell 可用（若從 Step 1 重新執行需重新定義）
CHECKPOINT_DIR = '/content/drive/MyDrive/cifar100_checkpoints'
BEST_CKPT  = os.path.join(CHECKPOINT_DIR, 'model_best.pth')
LAST_CKPT  = os.path.join(CHECKPOINT_DIR, 'checkpoint.pth')
MLFLOW_DIR = os.path.join(CHECKPOINT_DIR, 'mlruns')

# 優先使用 last checkpoint（含 optimizer state），其次 best checkpoint
if os.path.exists(LAST_CKPT):
    resume_path = LAST_CKPT
    print(f'🔄 發現 checkpoint，將從上次進度恢復：{resume_path}')
elif os.path.exists(BEST_CKPT):
    resume_path = BEST_CKPT
    print(f'🔄 發現 best checkpoint，將從此恢復：{resume_path}')
else:
    resume_path = ''
    print('🆕 未發現 checkpoint，從頭開始訓練')

resume_arg = f'--resume {resume_path}' if resume_path else ''

cmd = (
    f"python train.py "
    f"--epochs 200 "
    f"--batch-size 128 "
    f"--lr 0.1 "
    f"--num-workers 2 "
    f"--save-dir {CHECKPOINT_DIR} "
    f"--mlflow-dir {MLFLOW_DIR} "
    f"{resume_arg}"
)

print(f'\n🚀 執行指令：\n{cmd}\n')
!{cmd}

## Step 6｜確認 Checkpoint 已存到 Drive

In [ ]:
import os

CHECKPOINT_DIR = '/content/drive/MyDrive/cifar100_checkpoints'

print('📂 Drive checkpoint 目錄內容：')
files = sorted(os.listdir(CHECKPOINT_DIR))

if not files:
    print('  （目錄為空）')
else:
    for f in files:
        fpath = os.path.join(CHECKPOINT_DIR, f)
        if os.path.isfile(fpath):
            size = os.path.getsize(fpath) / 1024**2
            print(f'  {f:<30} ({size:.1f} MB)')
        else:
            print(f'  {f}/')

## Step 7｜執行評量（訓練完成後執行）

In [ ]:
import os

CHECKPOINT_DIR = '/content/drive/MyDrive/cifar100_checkpoints'
BEST_CKPT  = os.path.join(CHECKPOINT_DIR, 'model_best.pth')
EVAL_DIR   = os.path.join(CHECKPOINT_DIR, 'eval_results')

os.makedirs(EVAL_DIR, exist_ok=True)

if not os.path.exists(BEST_CKPT):
    print('⚠️  找不到 model_best.pth，請先完成訓練（Step 5）')
else:
    cmd = f"python evaluate.py --checkpoint {BEST_CKPT} --save-dir {EVAL_DIR}"
    print(f'🔍 執行評量：\n{cmd}\n')
    !{cmd}

    print('\n📂 評量結果：')
    for f in sorted(os.listdir(EVAL_DIR)):
        fpath = os.path.join(EVAL_DIR, f)
        size = os.path.getsize(fpath) / 1024**2
        print(f'  {f:<30} ({size:.1f} MB)')

## Step 8｜清除舊的 Checkpoint（需要時才執行）

> ⚠️ **警告**：執行此 cell 將刪除訓練進度，下次訓練將從頭開始。

In [ ]:
import os

CHECKPOINT_DIR = '/content/drive/MyDrive/cifar100_checkpoints'

# 安全確認
CONFIRM = True  # 改為 False 可防止意外執行

if not CONFIRM:
    print('⛔ 未確認，跳過刪除。請將 CONFIRM 改為 True 後再執行。')
else:
    targets = ['checkpoint.pth', 'model_best.pth', 'training_log.json']
    for f in targets:
        path = os.path.join(CHECKPOINT_DIR, f)
        if os.path.exists(path):
            os.remove(path)
            print(f'🗑️  已刪除：{f}')
        else:
            print(f'⏭️  不存在，跳過：{f}')
    print('\n✅ 完成，可以從頭訓練了')